# Import Statements

In [ ]:
import os
import time
from datetime import datetime
import random
import json
import pathlib
import math
import re

import pandas as pd
from pandas.io.formats import excel

import numpy as np

from google import genai
from google.genai import types
from google.genai.errors import ServerError

from pydantic import BaseModel, Field
from typing import List, Optional, Literal

from google.colab import userdata

# Config

In [ ]:
input_folder  = './'
output_folder = './'

submit_phase = 'test'

gemini_api_key = userdata.get('GEMINI_API_KEY')
gemini_model = "gemini-3-flash-preview"
gemini_seed = 42

file_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
study_list_file_path = f"{input_folder}{submit_phase}/{submit_phase}_articles.csv"
construct_definition_file_path = f"{input_folder}{submit_phase}/{submit_phase}_construct_definitions8.csv"
construct_valence_file_path = f"{input_folder}{submit_phase}/{submit_phase}_construct_valence.csv"
(study_list_file_path_noext, study_list_file_ext) = os.path.splitext(study_list_file_path)
submit_file_path = f"{input_folder}{submit_phase}_submit_{file_timestamp}{study_list_file_ext}"
study_pdf_folder_path = f"{input_folder}{submit_phase}/"

temp_output_path = output_folder + submit_phase + "_temp_{0}.xlsx"

index_col = "studyid"
construct_def_index_col = "Construct"
prompt_col = "prompt"
result_col = "result"
duration_col = "duration"
effect_mean_col = "effect_mean"
construct_definition_col = "Definition"
construct_negative_valence_col = "Negatively_valenced"
value_col = "aggregateeffectsize"
outlier_sd_col = "outlier_sd"
effect_count_col = "effect_count"

predictor_negative_col = "is_predictor_label_negative"
outcome_negative_col = "is_outcome_label_negative"
x_negative_col = "is_x_variable_label_negative"
x_reversed_col = "is_x_variable_reverse_scored"
y_negative_col = "is_y_variable_label_negative"
y_reversed_col = "is_y_variable_reverse_scored"

r_col = "r"
z_col = "z"
b_col = "b"
beta_col = "beta"
is_beta_col = "is_beta"
coeff_col = "coeff"
or_col = "oddsratio"
d_col = "d"
n_col = "n"
eta_squared_col = "eta_squared"
t_col = "t"
x_squared_col = "x_squared"
other_effect_col = "effect_size"

r_count_col = "r_count"
logit_count_col = "logit_count"
probit_count_col = "probit_count"
or_count_col = "or_count"
lreg_count_col = "lreg_count"
anova_count_col = "anova_count"
d_count_col = "d_count"
t_count_col = "t_count"
chi_count_col = "chi_count"
other_count_col = "other_count"

study_info_col = "info"
study_effects_col = "effects"
study_r_col = "r_coeff"
study_logit_col = "logit_coeff"
study_probit_col = "probit_coeff"
study_or_col = "oddsratios"
study_lreg_col = "lreg_coeff"
study_anova_col = "anova_eta"
study_d_col = "cohen_d"
study_t_col = "t_test"
study_chi_col = "chi_square"
study_othereffects_col = "other_effect"

submit_keep_cols = [value_col]

data_extraction_prompt = """# Task:
Today is [REPLACE_DATE], and you are a highly published social science professor who is a meta-analysis expert and routinely publishes in top-tier journals. Given the following research question, predictor variable description, and outcome variable description, extract relevant effect sizes reported in the attached research paper PDF file. Additionally, abide by the strict extraction rules.

# Research Question:
What is the bivariate association between '[REPLACE_CONSTRUCT1]' and '[REPLACE_CONSTRUCT2]'?

# Predictor Variable Description:
[REPLACE_PREDICTOR_DESCRIPTION]

# Outcome Variable Description:
[REPLACE_OUTCOME_DESCRIPTION]

# Strict Extraction Rules
- Extract effect sizes only when exactly one variable represents a form of '[REPLACE_CONSTRUCT1]' and the other represents a form of '[REPLACE_CONSTRUCT2]'. Exclude effects where both variables reflect '[REPLACE_CONSTRUCT1]' or both reflect '[REPLACE_CONSTRUCT2]'.
- Effect size pairs are analyzed cross-sectionally, therefore, the x and y variables must always be assessed or analyzed during the same wave, the same time period, or the same time point (e.g., Time1 X -> Time1 Y; Time2 X -> Time2 Y). If cross-lagged correlations are the only correlations available, extract them, such as when some variables are only measured once. If study variables are measured more than once, exclude cross-lagged pairs (Time1 X -> Time2 Y).
- Effect sizes may be found in tables, figures, footnotes, or within the paragraph text.
- Extract effect sizes as-is to include a negative sign if it exists.
- Tables containing effect sizes maybe rotated.
"""

class EffectInfo(BaseModel):
  page_number: int=Field(description="PDF page number where the coefficient was found.")
  x_variable_label: str=Field(description="Clearly interpretable x variable label as it appears in the study.")
  is_x_variable_label_negative: bool=Field(description="Indicates if the x variable label is negatively worded.")
  x_variable_description: str=Field(description="A brief description of the x variable and its scale.")
  is_x_variable_reverse_scored: bool=Field(description="Specifies whether higher numeric values, for the x variable, correspond to lower levels of the x variable label.")
  y_variable_label: str=Field(description="Clearly interpretable y variable label as it appears in the study.")
  is_y_variable_label_negative: bool=Field(description="Indicates if the y variable label is negatively worded.")
  y_variable_description: str=Field(description="A brief description of the y variable and its scale.")
  is_y_variable_reverse_scored: bool=Field(description="Specifies whether higher numeric values, for the y variable, correspond to lower levels of the y variable label.")
  model_description: str=Field(description="A concise and clearly understandable description of the type of linear model.")
  n: Optional[int]=Field(None, description="The number of observations or sample size for this study effect.")

class PearsonCorrelationCoefficient(EffectInfo):
  r: float=Field(description="The numeric Pearson's r Correlation or Zero-order Correlation coefficient.")

class LogisticRegressionCoefficient(EffectInfo):
  beta: float=Field(description="The numeric logistic regression, ordered logit, or logit coefficient, often labeled as coeff, beta, or the Greek symbol β.")

class ProbitRegressionCoefficient(EffectInfo):
  beta: float=Field(description="The numeric probability unit regression, ordered probit, or probit coefficient or average marginal effects, often labeled as coeff, beta, or the Greek symbol β.")

class OddsRatio(EffectInfo):
  oddsratio: float=Field(description="The numeric Odds Ratio value.")

class LinearRegression(EffectInfo):
  coeff: float=Field(description="The numeric coefficient, often labeled as the Greek symbol β, or in English beta, b, B, or coeff, for the linear regression model.")
  coeff_label: str=Field(description="Clearly interpretable label for coeff as it appears in the study.")
  is_beta: bool=Field(description="Indicates if the coeff label is beta or the Greek symbol β.")

class AnalysisOfVariance(EffectInfo):
  eta_squared: float=Field(description="The numeric effect size from an Analysis Of Variance (ANOVA) model, often labeled eta squared or partial eta squared.")

class CohenD(EffectInfo):
  d: float=Field(description="The numeric Cohen's d or Hedges' g effect size, often labeled 'd' or 'g'.")

class TTest(EffectInfo):
  t: float=Field(description="The numeric t-statisitc calculated from a t-test, often labeled 't'.")
  df: Optional[int]=Field(None, description="The numeric degrees of freedom for the t-test.")

class ChiSquareTest(EffectInfo):
  x_squared: float=Field(description="The numeric Chi-square statistic value.")

class OtherEffect(EffectInfo):
  effect_type_label: str=Field(description="Type of effect")
  effect_size: float=Field(description="The numeric effect size.")

class StudyEffects(BaseModel):
  r_coeff: List[PearsonCorrelationCoefficient]=Field(description="A list of extracted Pearson's r Correlation or Zero-order Correlation coefficients between the predictor and outcome variables.")
  logit_coeff: List[LogisticRegressionCoefficient]=Field(description="A list of extracted logistic regression, ordered logit, or logit coefficients between the predictor and outcome variables.")
  probit_coeff: List[ProbitRegressionCoefficient]=Field(description="A list of extracted probability unit regression, ordered probit, or probit coefficients or average marginal effects between the predictor and outcome variables.")
  oddsratios: List[OddsRatio]=Field(description="A list of extracted Odds Ratio values between the predictor and outcome variables.")
  lreg_coeff: List[LinearRegression]=Field(description="A list of extracted linear regression coefficients or betas between the predictor and outcome variables.")
  anova_eta: List[AnalysisOfVariance]=Field(description="A list of extracted Analysis Of Variance (ANOVA) model effect sizes between the predictor and outcome variables.")
  cohen_d: List[CohenD]=Field(description="A list of extracted Cohen's d or Hedges' g effect sizes between the predictor and outcome variables.")
  t_test: List[TTest]=Field(description="A list of extracted t-test statistic values between the predictor and outcome variables.")
  chi_square: List[ChiSquareTest]=Field(description="A list of extracted Chi-square statistic values between the predictor and outcome variables.")
  other_effect: List[OtherEffect]=Field(description="A list of all other relevant effect sizes.")

class StudyInfo(BaseModel):
  title: str=Field(description="Study title.")
  author: str=Field(description="Author(s) of the study.")
  total_pages: int=Field(description="Total number of pages in the PDF.")
  predictor_label: str=Field(description="Clearly interpretable research question predictor variable label as stated in the input prompt.")
  is_predictor_label_negative: bool=Field(description="Indicates if the research question predictor variable label is a negatively valenced construct.")
  outcome_label: str=Field(description="Clearly interpretable research question outcome variable label as stated in the input prompt.")
  is_outcome_label_negative: bool=Field(description="Indicates if the research question outcome variable label is a negatively valenced construct.")

class StudyDetails(BaseModel):
  info: StudyInfo=Field(description="Meta data about the study.")
  effects: StudyEffects=Field(description="Effect sizes extracted from the study.")


# Setup

In [ ]:
# Remove Sample Data folder
!rm -rf ./sample_data/

# Remove default Excel styling
excel.ExcelFormatter.header_style = None

# Initialize LLM AI API client
gemini_client = genai.Client(api_key=gemini_api_key)


# Load Submission Template

In [ ]:
df_src = pd.read_csv(study_list_file_path, encoding='cp1252').set_index(index_col)
df_src.dropna(how='all', axis=1, inplace=True)
df_construct_defs = pd.read_csv(construct_definition_file_path).set_index(construct_def_index_col)

df_src = df_src.join(df_construct_defs, on=f"{construct_def_index_col}1", how='left')
df_src.rename(columns={construct_definition_col: f"{construct_definition_col}1"}, inplace=True)
df_src = df_src.join(df_construct_defs, on=f"{construct_def_index_col}2", how='left')
df_src.rename(columns={construct_definition_col: f"{construct_definition_col}2"}, inplace=True)

if os.path.isfile(construct_valence_file_path):
  df_construct_valence = pd.read_csv(construct_valence_file_path).set_index(construct_def_index_col)
  df_src = df_src.join(df_construct_valence, on=f"{construct_def_index_col}1", how='left')
  df_src.rename(columns={construct_negative_valence_col: f"{construct_negative_valence_col}1"}, inplace=True)
  df_src = df_src.join(df_construct_valence, on=f"{construct_def_index_col}2", how='left')
  df_src.rename(columns={construct_negative_valence_col: f"{construct_negative_valence_col}2"}, inplace=True)

df_out = df_src.copy()

#df_out.head()

# Process Studies

In [ ]:
print("Start processing studies...")
def extract_effect_sizes(study_pdf_file_path, construct1, construct2, constructdef1, constructdef2):
  response_text = ""
  temp_prompt = ""
  if os.path.isfile(study_pdf_file_path):
    print(f"{study_pdf_file_path}...")

    temp_prompt = data_extraction_prompt.replace("[REPLACE_DATE]", str(datetime.now().strftime("%A, %B %d, %Y at %H:%M:%S.%f")))
    temp_prompt = temp_prompt.replace("[REPLACE_CONSTRUCT1]", construct1)
    temp_prompt = temp_prompt.replace("[REPLACE_CONSTRUCT2]", construct2)
    temp_prompt = temp_prompt.replace("[REPLACE_PREDICTOR_DESCRIPTION]", constructdef1)
    temp_prompt = temp_prompt.replace("[REPLACE_OUTCOME_DESCRIPTION]", constructdef2)

    #print(temp_prompt)

    try:
      pdf_pathlib = pathlib.Path(study_pdf_file_path)

      response = gemini_client.models.generate_content(
          model=gemini_model,
          contents=types.Content(
              parts=[
                  types.Part(text=temp_prompt),
                  types.Part.from_bytes(
                    data=pdf_pathlib.read_bytes(),
                    mime_type="application/pdf"
                  ),
              ],
              role='user',
          ),
          config=types.GenerateContentConfig(
              seed=gemini_seed,
              thinking_config=types.ThinkingConfig(
                thinking_level=types.ThinkingLevel.HIGH
              ),
              response_mime_type="application/json",
              response_json_schema=StudyDetails.model_json_schema(),
          )
      )

      response_text = response.text

      # Excel spreadsheet cell has a 32,767 character length limit
      if len(response_text) > (1024*32-1):
        response_text = re.sub(r' +', ' ', response_text)
        response_text = re.sub(r'\s*\"[xy]_variable_description\".*\"\,\s*', '', response_text)

    except Exception as e:
      response_text = f"GenAI request error for {i}: {e}"

    #print(f"Response Text: {response_text}")

    time.sleep(random.randint(10, 15))

  else:
    print(f"{study_pdf_file_path}... missing.")

  return (temp_prompt, response_text)

for i, r in df_src.iterrows():
  study_file_name = f"{i}.pdf"
  study_file_path = f"{study_pdf_folder_path}{study_file_name}"

  start_time = time.time()
  (temp_prompt, response_text) = extract_effect_sizes(study_file_path, r.Construct1, r.Construct2, r.Definition1, r.Definition2)
  end_time = time.time()

  df_out.loc[df_out.index == i, [prompt_col]] = temp_prompt
  df_out.loc[df_out.index == i, [result_col]] = response_text
  df_out.loc[df_out.index == i, [duration_col]] = str(round(end_time - start_time, 1))

df_out.to_excel(temp_output_path.format("processed"), index=True)

print("Completed.")

#df_out.head()

Start processing studies...
./test/study1.pdf...
./test/study2.pdf... missing.
./test/study3.pdf... missing.
./test/study4.pdf... missing.
./test/study5.pdf... missing.
./test/study6.pdf... missing.
./test/study7.pdf... missing.
./test/study8.pdf... missing.
./test/study9.pdf... missing.
./test/study10.pdf... missing.
./test/study11.pdf... missing.
./test/study12.pdf... missing.
./test/study13.pdf... missing.
./test/study14.pdf... missing.
./test/study15.pdf... missing.
./test/study16.pdf... missing.
./test/study17.pdf... missing.
./test/study18.pdf... missing.
./test/study19.pdf... missing.
./test/study20.pdf... missing.
./test/study21.pdf... missing.
./test/study22.pdf... missing.
./test/study23.pdf... missing.
./test/study24.pdf... missing.
./test/study25.pdf... missing.
./test/study26.pdf... missing.
./test/study27.pdf... missing.
./test/study28.pdf... missing.
./test/study29.pdf... missing.
./test/study30.pdf... missing.
./test/study31.pdf... missing.
./test/study32.pdf... missing

# Post-Processing

In [ ]:
print("Start post-processing...")
processed_file_path = temp_output_path.format("processed")

if os.path.isfile(processed_file_path):
  df_src = pd.read_excel(processed_file_path).set_index(index_col)
  df_out = df_src.copy()

def count_effects(js, js_effects_key, effect_col):
  val_list = []

  if js:
    if js_effects_key in js[study_effects_col]:
      for e in js[study_effects_col][js_effects_key]:
        if e:
          val_list.append(e[effect_col])

  return val_list

def process_regression_coeff(js, js_effects_key, effect_col, is_construct1_negative, is_construct2_negative):
  beta_list = []

  if js:
    if js_effects_key in js[study_effects_col]:
      for e in js[study_effects_col][js_effects_key]:
        if e and e[is_beta_col]:
          beta = e[effect_col]

          if(is_construct1_negative == is_construct2_negative):
            if((e[x_negative_col] ^ e[x_reversed_col]) != (e[y_negative_col] ^ e[y_reversed_col])):
              beta = (-1*beta)
          else:
            if((e[x_negative_col] ^ e[x_reversed_col]) == (e[y_negative_col] ^ e[y_reversed_col])):
              beta = (-1*beta)

          beta_list.append(beta)

  return beta_list

for i, r in df_out.iterrows():
  effect_mean_val = np.nan
  r_count = 0
  logit_count = 0
  probit_count = 0
  or_count = 0
  lreg_count = 0
  anova_count = 0
  d_count = 0
  t_count = 0
  chi_count = 0
  effect_count = 0
  other_count = 0
  all_r_values = []

  json_result = None
  try:
    json_result = json.loads(r[result_col])
  except Exception as e:
    print(f"Missing study {i}...")

  if json_result:
    print(f"Aggregate results for {i}...")

    if os.path.isfile(construct_valence_file_path):
      is_construct1_negatively_valenced = r[f"{construct_negative_valence_col}1"]
      is_construct2_negatively_valenced = r[f"{construct_negative_valence_col}2"]
    else:
      is_construct1_negatively_valenced = json_result[study_info_col][predictor_negative_col]
      is_construct2_negatively_valenced = json_result[study_info_col][outcome_negative_col]

    ##########################
    ## Process Pearson's r
    ##########################
    r_values = []
    for e in json_result[study_effects_col][study_r_col]:
      if e:
        r_val = e[r_col]

        if(is_construct1_negatively_valenced == is_construct2_negatively_valenced):
          if((e[x_negative_col] ^ e[x_reversed_col]) != (e[y_negative_col] ^ e[y_reversed_col])):
            r_val = (-1*r_val)
        else:
          if((e[x_negative_col] ^ e[x_reversed_col]) == (e[y_negative_col] ^ e[y_reversed_col])):
            r_val = (-1*r_val)

        r_values.append(r_val)

    #print(f"r: {r_values}")

    r_count = len(r_values)

    if r_count > 0:
      all_r_values.extend(r_values)

    ##########################
    ## Process Log Odds Ratios
    ##########################
    logit_values = count_effects(json_result, study_logit_col, beta_col)
    logit_count = len(logit_values)

    ##########################
    ## Process Probit Log Odds Ratios
    ##########################
    probit_values = count_effects(json_result, study_probit_col, beta_col)
    probit_count = len(probit_values)

    ##########################
    ## Process Odds Ratios
    ##########################
    or_values = count_effects(json_result, study_or_col, or_col)
    or_count = len(or_values)

    ##########################
    ## Process Linear Regression
    ##########################
    lreg_values = count_effects(json_result, study_lreg_col, coeff_col)
    #lreg_values = process_regression_coeff(json_result, study_lreg_col, coeff_col, is_construct1_negatively_valenced, is_construct2_negatively_valenced)
    lreg_count = len(lreg_values)

    #print(f"lreg: {lreg_values}")

    #if lreg_count > 0:
    #  all_r_values.extend(lreg_values)

    ##########################
    ## Process ANOVA model
    ##########################
    anova_values = count_effects(json_result, study_anova_col, eta_squared_col)
    anova_count = len(anova_values)

    ##########################
    ## Process Cohen's d
    ##########################
    d_values = count_effects(json_result, study_d_col, d_col)
    d_count = len(d_values)

    ##########################
    ## Process t-test
    ##########################
    t_values = count_effects(json_result, study_t_col, t_col)
    t_count = len(t_values)

    ##########################
    ## Process Chi-square
    ##########################
    chi_values = count_effects(json_result, study_chi_col, x_squared_col)
    chi_count = len(chi_values)

    ##########################
    ## Report Other Effects
    ##########################
    other_values = count_effects(json_result, study_othereffects_col, other_effect_col)
    other_count = len(other_values)

    ##########################
    ## All Effects
    ##########################
    effect_count = r_count

    if effect_count > 0:
      #print(f"All r: {all_r_values}")
      effect_mean_val = round(np.mean(all_r_values), 4)

    effect_count = r_count + logit_count + probit_count + or_count + lreg_count + anova_count + d_count + t_count + chi_count + other_count

    #print(f"study mean: {effect_mean_val}")

  df_out.loc[df_out.index == i, [effect_mean_col]] = effect_mean_val
  df_out.loc[df_out.index == i, [r_count_col]] = r_count
  df_out.loc[df_out.index == i, [logit_count_col]] = logit_count
  df_out.loc[df_out.index == i, [probit_count_col]] = probit_count
  df_out.loc[df_out.index == i, [or_count_col]] = or_count
  df_out.loc[df_out.index == i, [lreg_count_col]] = lreg_count
  df_out.loc[df_out.index == i, [anova_count_col]] = anova_count
  df_out.loc[df_out.index == i, [d_count_col]] = d_count
  df_out.loc[df_out.index == i, [t_count_col]] = t_count
  df_out.loc[df_out.index == i, [chi_count_col]] = chi_count
  df_out.loc[df_out.index == i, [other_count_col]] = other_count
  df_out.loc[df_out.index == i, [effect_count_col]] = effect_count

df_out[value_col] = df_out[effect_mean_col]

df_out.to_excel(temp_output_path.format("post_processed"), index=True)

print("Completed.")

#df_out.head()

Start post-processing...
Aggregate results for study1...
Missing study study2...
Missing study study3...
Missing study study4...
Missing study study5...
Missing study study6...
Missing study study7...
Missing study study8...
Missing study study9...
Missing study study10...
Missing study study11...
Missing study study12...
Missing study study13...
Missing study study14...
Missing study study15...
Missing study study16...
Missing study study17...
Missing study study18...
Missing study study19...
Missing study study20...
Missing study study21...
Missing study study22...
Missing study study23...
Missing study study24...
Missing study study25...
Missing study study26...
Missing study study27...
Missing study study28...
Missing study study29...
Missing study study30...
Missing study study31...
Missing study study32...
Missing study study33...
Missing study study34...
Missing study study35...
Missing study study36...
Missing study study37...
Missing study study38...
Missing study study39...
M

# Submission File

In [ ]:
print("Create submission file...")
df_submit = df_out[submit_keep_cols].copy()
df_submit.to_csv(submit_file_path, index=True)
print("Completed.")

#df_submit.head()

Create submission file...
Completed.
